In [ ]:
import torch
run_config = {}
# Plot callback
total_n_valid = data_loader.valid_ds.aligndata.shape[0]
val_data = [data_loader.valid_ds.__getitem__(idx) for idx in range(0, total_n_valid, total_n_valid//4)]
if run_config['paths']['hmi_path'] is not None and run_config['paths']['aia_path'] is not None:
    hmi_val = torch.tensor(np.array([val_data[idx][0] for idx, _ in enumerate(val_data)]))    
    aia_val = torch.tensor(np.array([val_data[idx][1] for idx, _ in enumerate(val_data)]))
    eve_val = torch.tensor(np.array([val_data[idx][2] for idx, _ in enumerate(val_data)]))
    # stacking images
    hmi_aia_stack = torch.cat((hmi_val, aia_val), axis=0)
    hmi_aia_config = run_config["sci_parameters"]["components"] + run_config["sci_parameters"]["aia_wavelengths"] 
    image_callback = ImagePredictionLogger(hmi_aia_stack, eve_val, run_config["sci_parameters"]["eve_ions"], hmi_aia_config)
    
elif run_config['paths']['hmi_path'] is not None and run_config['paths']['aia_path'] is None:
    flag = 'HMI'
    hmi_val = torch.tensor(np.array([val_data[idx][0] for idx, _ in enumerate(val_data)]))    
    eve_val = torch.tensor(np.array([val_data[idx][1] for idx, _ in enumerate(val_data)]))
    image_callback = ImagePredictionLogger(hmi_val, eve_val, run_config["sci_parameters"]["eve_ions"], run_config["sci_parameters"]["components"],flag)

else: # HMI is None but AIA is not  
    flag = 'AIA'
    aia_val = torch.tensor(np.array([val_data[idx][0] for idx, _ in enumerate(val_data)]))
    eve_val = torch.tensor(np.array([val_data[idx][1] for idx, _ in enumerate(val_data)]))
    image_callback = ImagePredictionLogger(aia_val, eve_val, run_config["sci_parameters"]["eve_ions"], run_config["sci_parameters"]["aia_wavelengths"],flag)

In [ ]:

aia = torch.randn(9,4,4)
hmi = torch.randn(3,4,4)
aiahmi= torch.cat((aia,hmi),axis=0)

In [ ]:
class ImagePredictionLoggerHMI(Callback):

    def __init__(self, val_imgs, val_eve, ions, channels):
        super().__init__()
                    
        self.val_eve = val_eve
        self.names = ions

        self.val_imgs, self.val_eve = val_imgs, val_eve
        
        # either HMI + AIA or any of them individually
        self.channels = channels

    def on_validation_epoch_end(self, trainer, pl_module):
        # Bring the tensors to CPU
        val_imgs = self.val_imgs.to(device=pl_module.device)
        # Get model prediction
        # pred_eve = pl_module.forward(val_imgs).cpu().numpy()
        pred_eve = pl_module.forward_unnormalize(val_imgs).cpu().numpy()
        val_eve = unnormalize(self.val_eve, pl_module.eve_norm).numpy()
        val_imgs = val_imgs.cpu().numpy()

        # create matplotlib figure
        fig = self.plot_channel_eve(val_imgs, val_eve, pred_eve)
        # Log the images to wandb
        trainer.logger.experiment.log({"AIA Images and EVE bar plots": wandb.Image(fig)})
        plt.close(fig)

    def plot_channel_eve(self, val_imgs, val_eve, pred_eve):
        """
        Function to plot a 4 channel AIA stack and the EVE barplots

        Arguments:
        ----------
            val_imgs: numpy array
                Stack with 4 image channels
            val_eve: numpy array
                Stack of ground-truth eve channels
            pred_eve: numpy array
                Stack of predicted eve channels
        Returns:
        --------
            fig: matplotlib figure
                figure with plots
        """
        samples = pred_eve.shape[0]
        n_channels = len(self.channels)

        cmap_dict = {
            'By': 'hmimag',
            'Bz': 'hmimag',
            'Bx': 'hmimag',
            "131A": 'sdoaia131',
            "1600A": 'sdoaia1600',
            "1700A": 'sdoaia1700',
            "171A": 'sdoaia171',
            "193A": 'sdoaia193',
            "211A": 'sdoaia211',
            "304A": 'sdoaia304',
            "335A": 'sdoaia335',
            "94A": 'sdoaia94'}

        fig = plt.figure(figsize=(16,12), dpi=1200)
        nrows = 4
        ncols = 4

        # Bx
        plt.subplot(nrows,ncols,0)
        plt.imshow(val_imgs[0], cmap = plt.get_cmap(cmap_dict['Bx']), vmin = 0, vmax = 1)
        plt.title('Bx')

        # By
        plt.subplot(nrows,ncols,1)
        plt.imshow(val_imgs[1], cmap = plt.get_cmap(cmap_dict['By']), vmin = 0, vmax = 1)
        plt.title('By')

        # Bz
        plt.subplot(nrows,ncols,2)
        plt.imshow(val_imgs[2], cmap = plt.get_cmap(cmap_dict['Bz']), vmin = 0, vmax = 1)
        plt.title('Bz')

        # 131A
        plt.subplot(nrows,ncols,3)
        plt.imshow(val_imgs[3], cmap = plt.get_cmap(cmap_dict['131A']), vmin = 0, vmax = 1)
        plt.title('131A')

        # 1600A
        plt.subplot(nrows,ncols,4)
        plt.imshow(val_imgs[4], cmap = plt.get_cmap(cmap_dict['1600A']), vmin = 0, vmax = 1)
        plt.title('1600A')

        # 1700A
        plt.subplot(nrows,ncols,5)
        plt.imshow(val_imgs[5], cmap = plt.get_cmap(cmap_dict['1700A']), vmin = 0, vmax = 1)
        plt.title('1700A')

        # 171A
        plt.subplot(nrows,ncols,6)
        plt.imshow(val_imgs[6], cmap = plt.get_cmap(cmap_dict['171A']), vmin = 0, vmax = 1)
        plt.title('171A')

        # 193A
        plt.subplot(nrows,ncols,7)
        plt.imshow(val_imgs[7], cmap = plt.get_cmap(cmap_dict['193A']), vmin = 0, vmax = 1)
        plt.title('193A')

        # 211A
        plt.subplot(nrows,ncols,8)
        plt.imshow(val_imgs[8], cmap = plt.get_cmap(cmap_dict['211A']), vmin = 0, vmax = 1)
        plt.title('211A')

        # 304A
        plt.subplot(nrows,ncols,9)
        plt.imshow(val_imgs[9], cmap = plt.get_cmap(cmap_dict['304A']), vmin = 0, vmax = 1)
        plt.title('304A')

        # 335A
        plt.subplot(nrows,ncols,10)
        plt.imshow(val_imgs[10], cmap = plt.get_cmap(cmap_dict['335A']), vmin = 0, vmax = 1)
        plt.title('335A')

        # 94A
        plt.subplot(nrows,ncols,11)
        plt.imshow(val_imgs[11], cmap = plt.get_cmap(cmap_dict['94A']), vmin = 0, vmax = 1)
        plt.title('94A')

        # EVE
        plt.subplot(nrows,ncols,12)
        plt.bar(np.arange(0,len(val_eve[:,:])), val_eve[:,:], label='ground truth')
        plt.bar(np.arange(0,len(pred_eve[s,:])), pred_eve[s,:], width = 0.5, label='prediction', alpha=0.5)

        plt.show()


        if n_aia_wavelengths < 3:
            nrows = 1
            ncols = n_aia_wavelengths
            fig = plt.figure(figsize=( 9+9/4*n_aia_wavelengths, 3*samples), dpi=150)
            gs = fig.add_gridspec(samples, n_aia_wavelengths+3, wspace=0.4, hspace=0.25)
        elif n_aia_wavelengths < 5:
            nrows = 2
            ncols = 2
            fig = plt.figure(figsize=( 9+9/4*2, 6*samples), dpi=150)
            gs = fig.add_gridspec(2*samples, 5, wspace=0.4, hspace=0.25)
        elif n_aia_wavelengths < 7:
            nrows = 2
            ncols = 3
            fig = plt.figure(figsize=( 9+9/4*3, 6*samples), dpi=150)
            gs = fig.add_gridspec(2*samples, 6, wspace=0.4, hspace=0.25)
        else:
            nrows = 2
            ncols = 4
            fig = plt.figure(figsize=( 18, 6*samples), dpi=150)
            gs = fig.add_gridspec(2*samples, 7, wspace=0.4, hspace=0.25)
        
        cmaps_all = [f"sdoaia{wavelength.split('A')[0]}" for wavelength in self.aia_wavelengths]
        
        cmaps = [cmaps_all[i] for i, _ in enumerate(self.aia_wavelengths)]
        n_plots = 0

        for s in range(samples):
            for i in range(nrows):
                for j in range(ncols):
                    if n_plots < n_aia_wavelengths: 
                        ax = fig.add_subplot(gs[s*nrows+i, j])
                        ax.imshow(val_imgs[s, i*ncols+j], cmap = plt.get_cmap(cmaps[i*ncols+j]), vmin = 0, vmax = 1)
                        ax.text(0.01, 0.99, cmaps[i*ncols+j], horizontalalignment='left', verticalalignment='top', color = 'w', transform=ax.transAxes)
                        ax.set_axis_off()
                        n_plots += 1
            n_plots = 0
            #eve data
            ax5 = fig.add_subplot(gs[s*nrows, ncols:])
            ax5.bar(np.arange(0,len(val_eve[s,:])), val_eve[s,:], label='ground truth')
            ax5.bar(np.arange(0,len(pred_eve[s,:])), pred_eve[s,:], width = 0.5, label='prediction', alpha=0.5)
            ax5.set_xticks(np.arange(0,len(val_eve[s,:])))
            ax5.set_xticklabels(self.names,rotation = 45)
            ax5.set_yscale('log')
            ax5.legend()

        # fig.tight_layout()
        return fig


Test cells

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import sunpy.visualization.colormaps

def plot_channel_eve(val_imgs, val_eve, pred_eve, channels):
    """
    Function to plot a 4 channel AIA stack and the EVE barplots

    Arguments:
    ----------
        val_imgs: numpy array
            Stack with 4 image channels
        val_eve: numpy array
            Stack of ground-truth eve channels
        pred_eve: numpy array
            Stack of predicted eve channels
    Returns:
    --------
        fig: matplotlib figure
            figure with plots
    """
    samples = pred_eve.shape[0]
    n_channels = len(channels)

    cmap_dict = {
        'By': 'hmimag',
        'Bz': 'hmimag',
        'Bx': 'hmimag',
        "131A": 'sdoaia131',
        "1600A": 'sdoaia1600',
        "1700A": 'sdoaia1700',
        "171A": 'sdoaia171',
        "193A": 'sdoaia193',
        "211A": 'sdoaia211',
        "304A": 'sdoaia304',
        "335A": 'sdoaia335',
        "94A": 'sdoaia94'}

    fig = plt.figure(figsize=(16,12), dpi=1200)
    nrows = 4
    ncols = 4

    # Bx
    plt.subplot(nrows,ncols,1)
    plt.imshow(val_imgs[0], cmap = plt.get_cmap(cmap_dict['Bx']), vmin = 0, vmax = 1)
    plt.title('Bx')

    # By
    plt.subplot(nrows,ncols,2)
    plt.imshow(val_imgs[1], cmap = plt.get_cmap(cmap_dict['By']), vmin = 0, vmax = 1)
    plt.title('By')

    # Bz
    plt.subplot(nrows,ncols,3)
    plt.imshow(val_imgs[2], cmap = plt.get_cmap(cmap_dict['Bz']), vmin = 0, vmax = 1)
    plt.title('Bz')

    # 131A
    plt.subplot(nrows,ncols,4)
    plt.imshow(val_imgs[3], cmap = plt.get_cmap(cmap_dict['131A']), vmin = 0, vmax = 1)
    plt.title('131A')

    # 1600A
    plt.subplot(nrows,ncols,5)
    plt.imshow(val_imgs[4], cmap = plt.get_cmap(cmap_dict['1600A']), vmin = 0, vmax = 1)
    plt.title('1600A')

    # 1700A
    plt.subplot(nrows,ncols,6)
    plt.imshow(val_imgs[5], cmap = plt.get_cmap(cmap_dict['1700A']), vmin = 0, vmax = 1)
    plt.title('1700A')

    # 171A
    plt.subplot(nrows,ncols,7)
    plt.imshow(val_imgs[6], cmap = plt.get_cmap(cmap_dict['171A']), vmin = 0, vmax = 1)
    plt.title('171A')

    # 193A
    plt.subplot(nrows,ncols,8)
    plt.imshow(val_imgs[7], cmap = plt.get_cmap(cmap_dict['193A']), vmin = 0, vmax = 1)
    plt.title('193A')

    # 211A
    plt.subplot(nrows,ncols,9)
    plt.imshow(val_imgs[8], cmap = plt.get_cmap(cmap_dict['211A']), vmin = 0, vmax = 1)
    plt.title('211A')

    # 304A
    plt.subplot(nrows,ncols,10)
    plt.imshow(val_imgs[9], cmap = plt.get_cmap(cmap_dict['304A']), vmin = 0, vmax = 1)
    plt.title('304A')

    # 335A
    plt.subplot(nrows,ncols,11)
    plt.imshow(val_imgs[10], cmap = plt.get_cmap(cmap_dict['335A']), vmin = 0, vmax = 1)
    plt.title('335A')

    # 94A
    plt.subplot(nrows,ncols,12)
    plt.imshow(val_imgs[11], cmap = plt.get_cmap(cmap_dict['94A']), vmin = 0, vmax = 1)
    plt.title('94A')

    # EVE
    plt.subplot(nrows,1,nrows)
    plt.bar(np.arange(0,len(val_eve)), val_eve[:], label='ground truth')
    plt.bar(np.arange(0,len(pred_eve)), pred_eve[:], width = 0.5, label='prediction', alpha=0.5)

    plt.show()



In [ ]:
aia = torch.randn(9,4,4)
hmi = torch.randn(3,4,4)
aiahmi= torch.cat((aia,hmi),axis=0)

# in my mind these should be like single values? Not sure what w%b is polotting, is it a single time step prediction? if so, we can get this by
# squeezing the input
eve_val = np.random.randint(1,10,10)
eve_pred = np.random.randint(1,10,10)

chan = ["Bx", "By", "Bz"]+["131A", "1600A", "1700A", "171A", "193A", "211A", "304A", "335A", "94A"]

plot_channel_eve(aiahmi, eve_val, eve_pred, chan)